# midsphere Python SDK — full walkthrough

This notebook exercises **every** capability of the `midsphere` Python SDK against a
running backend: agents, end users, tools, knowledge, running an agent with live
event streaming, the in-sandbox debug console, ratings, and error handling.

**Prerequisites**
- The backend is running (here: `http://127.0.0.1:8010`).
- You have a **workspace API key** (`ms_...`) — created in the dashboard or via the
  `createApiKey` GraphQL mutation.

Install the SDK with `pip install midsphere`. Every call is `async`; this notebook
uses top-level `await`, which Jupyter supports natively.

## 1. Connect

Initialize the client with your API key. `base_url` defaults to the hosted API; override it for local/self-hosted backends.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path("midsphere.env"))   # config written by dev-up.sh

from midsphere import (
    Midsphere,
    MidsphereError,        # base class
    MidsphereUserError,    # domain failure (bad input, quota, permission)
    MidsphereApiError,     # transport / system failure (auth, rate limit, network)
    MidsphereStreamError,  # failure talking directly to the sandbox stream
)

API_KEY = os.environ["MIDSPHERE_API_KEY"]
BASE_URL = os.environ.get("MIDSPHERE_BASE_URL", "http://localhost:8010")

client = Midsphere(api_key=API_KEY, base_url=BASE_URL)
client

## 2. Health & the raw GraphQL escape hatch

Anything not yet wrapped by a typed resource is reachable through `client.gql.request(...)`.

In [ ]:
await client.gql.request("{ health }")

## 3. Agents

List the workspace's agents, fetch one by id, or bind to one for repeated runs.

In [ ]:
agents = await client.agents.list(first=25)   # Relay-style pagination
for a in agents:
    print(f"{a.id}  {a.name:<18} tier={a.default_model_tier}  active={a.is_active}")

agent = agents[0]   # we'll use this one throughout

In [ ]:
# Fetch a single agent by its global id (returns None if not found).
await client.agents.get(agent.id)

In [ ]:
# Bind to an agent — AgentRef gives you .run(...) and .get() scoped to it.
agent_ref = client.agent(agent.id)
agent_ref

## 4. End users

An *end user* is the developer's user — the person an agent acts on behalf of. You pass
ids from your own auth system; `ensure(...)` resolves-or-creates idempotently.

In [ ]:
user = await client.end_users.ensure(
    external_user_id="auth0|ada-12345",   # your system's id
    display_name="Ada Lovelace",
    tier_label="pro",
)
user

In [ ]:
# Per-user controls: credit balance, max concurrency, allowed model tier.
await client.end_users.set_controls(
    id=user.id,
    credits=500,
    concurrency_max=3,
    model_tier="pro",
)

In [ ]:
# Look up by your external id, list the workspace's end users, and update a field.
print("by external id:", await client.end_users.get_by_external_id("auth0|ada-123"))
print("list          :", await client.end_users.list(first=10))
await client.end_users.update(id=user.id, display_name="Ada L.")

## 5. Tools

Tools are the callables an agent may invoke. Lifecycle is **activate / deactivate** —
there is no hard delete.

In [ ]:
# Create is idempotent here: on re-run we catch the duplicate and reuse the existing tool.
try:
    tool = await client.tools.create(
        name="search_orders",
        description="Look up a customer order by id",
        parameters_schema={
            "type": "object",
            "properties": {"orderId": {"type": "string"}},
            "required": ["orderId"],
        },
        execution_endpoint="https://api.example.com/tools/search-orders",
    )
    print("created:", tool.id)
except MidsphereUserError as e:
    tool = (await client.tools.list())[0]
    print(f"already exists ({e}); reusing {tool.id}")

In [ ]:
print("get      :", await client.tools.get(tool.id))
await client.tools.update(id=tool.id, description="Look up an order by its id")
await client.tools.deactivate(tool.id)          # soft-disable
await client.tools.activate(tool.id)            # re-enable
print("active   :", len(await client.tools.list(is_active=True)), "tool(s)")

## 6. Knowledge

Knowledge bases ground the agent. Kinds: `document`, `dataset`, `connected_source`.
A `connected_source` points at a URL (no file upload needed).

In [ ]:
try:
    kb = await client.knowledge.create(
        name="Returns policy",
        kind="connected_source",
        description="Company returns + refunds policy",
        source_endpoint="https://example.com/policies/returns",
    )
    print("created:", kb.id)
except MidsphereUserError as e:
    existing = await client.knowledge.list()
    kb = existing[0] if existing else None
    print(f"create skipped ({e}); reusing {kb.id if kb else None}")

In [ ]:
if kb is not None:
    await client.knowledge.update(id=kb.id, description="Returns + refunds")
    await client.knowledge.reindex(kb.id)       # re-index the connected source
    await client.knowledge.deactivate(kb.id)
    await client.knowledge.activate(kb.id)
print("knowledge bases:", await client.knowledge.list())

## 7. Run an agent + stream events

`client.run(...)` starts a run in an isolated sandbox and returns a `Run` handle.
`run.stream()` is an async generator that yields the sandbox's events **in order**,
returning after the terminal event (`run_completed` / `run_failed` / `run_cancelled`).

In [ ]:
run = await client.run(
    agent_id=agent.id,
    end_user_id=user.id,
    prompt="What is 6 times 7? Answer in one short sentence.",
)
print("run id:", run.id, "| uid:", run.uid, "| status:", run.status)

events = []
async for event in run.stream():
    events.append(event.type)
    print(f"  → {event.type}: {str(event.payload)[:140]}")

print("\nevent sequence:", events)
print("final status   :", run.status)

## 8. The in-sandbox debug console (port 4321)

The agent runs inside an E2B sandbox that also serves an **Astro debug console** on
port `4321`. The SDK derives its URL from the run's sandbox host — handy for inspecting
a live run.

In [ ]:
print("sandbox stream host :", run.sandbox_url)
print("debug console (4321) :", run.debug_console_url)

In [ ]:
# The same mapping straight from the E2B SDK: the sandbox id is the host's <id> segment,
# and get_host(4321) returns the per-sandbox proxy host for the console port.
from urllib.parse import urlsplit

try:
    from e2b import AsyncSandbox
    E2B_API_KEY = os.environ.get("E2B_API_KEY", "")
    sandbox_id = urlsplit(run.sandbox_url).netloc.split("-", 1)[1].split(".")[0]
    sbx = await AsyncSandbox.connect(sandbox_id, api_key=E2B_API_KEY)
    print("console via e2b sdk :", f"https://{sbx.get_host(4321)}")
except Exception as e:
    # The sandbox may already be gone if the run finished/terminated.
    print("e2b connect skipped:", type(e).__name__, e)

## 9. Run lifecycle controls

Beyond streaming, a `Run` exposes:

| call | purpose |
|---|---|
| `await run.wait()` | consume the stream and return a structured `RunOutcome` |
| `await run.refresh_token()` | mint a fresh capability token mid-run (long runs) |
| `await run.cancel()` | stop an in-flight run |
| `await run.respond(request_id, decision)` | answer a human-in-the-loop (HITL) request |

HITL pattern — when the agent needs approval it emits a `hitl_requested` event:

```python
async for event in run.stream():
    if event.type == "hitl_requested":
        await run.respond(event.payload["request_id"], "approve")
```

In [ ]:
# token_expires_at reflects the current capability token's lifetime.
print("capability token expires at:", run.token_expires_at)

## 10. Ratings

End users rate **completed** runs 1–5.

In [ ]:
try:
    await client.ratings.submit(run_uid=run.uid, score=5, comment="Spot on")
    print("rating submitted")
except MidsphereUserError as e:
    print("rating rejected (run not completed):", e)

## 11. Error handling

Domain failures raise `MidsphereUserError` (carrying a stable `code`); transport/system
failures raise `MidsphereApiError`; sandbox-stream failures raise `MidsphereStreamError`.
All extend `MidsphereError`.

In [ ]:
try:
    await client.end_users.set_controls(id="not-a-real-id", credits=10)
except MidsphereError as e:
    print(f"caught {type(e).__name__}: {e}")

## 12. Cleanup

Close the underlying HTTP client (or use `async with Midsphere(...) as client:`).

In [ ]:
await client.aclose()
print("client closed")